# 00 - Validate Admin Monitoring access

This read-only notebook verifies that the current Fabric identity and capacity can support the planned Admin Monitoring archive.

It checks:

1. The source and target workspaces resolve by exact name.
2. The notebook is running in the intended target workspace.
3. All four semantic models resolve by exact name.
4. A minimal DAX query can execute against each model.
5. Table and column metadata can be read through Semantic Link/TOM.
6. One row can be queried from a discovered table in each model.

**This notebook does not create a Lakehouse or write any data.** A failure in TOM metadata access is treated as blocking because later notebooks need reliable schema discovery to archive every accessible table.

## Planned notebook sequence

- **00** - Validate access and extraction prerequisites (this notebook)
- **01** - Create or resolve the destination Lakehouse and initialize audit/control tables
- **02** - Capture the initial 30-day baseline with reconciliation and restart safety
- **03** - Run the daily incremental snapshot with overlap, deduplication, and idempotent writes
- **04** - Reconcile coverage, detect schema drift, and report missing or late-arriving data


In [ ]:
# Parameters - a Fabric pipeline can override these values later.
source_workspace_name = "Admin monitoring"
target_workspace_name = "YOUR_TARGET_WORKSPACE"

semantic_model_names = [
    "Content Sharing",
    "Feature Usage and Adoption",
    "OneLake Catalog - Governance for Admins",
    "Purview Hub",
]

require_target_workspace_context = True
sample_rows_per_model = 1
max_error_characters = 1500


In [ ]:
from datetime import datetime, timezone
from uuid import UUID

import pandas as pd
import requests
import sempy.fabric as fabric


def normalized_name(value):
    return "".join(character for character in str(value).casefold() if character.isalnum())


def find_column(dataframe, candidates, purpose, required=True):
    normalized_columns = {normalized_name(column): column for column in dataframe.columns}
    for candidate in candidates:
        match = normalized_columns.get(normalized_name(candidate))
        if match is not None:
            return match

    if required:
        raise KeyError(
            f"Could not identify the {purpose} column. Available columns: "
            f"{list(dataframe.columns)}"
        )
    return None


def exact_name_matches(dataframe, name_column, expected_name):
    return dataframe[
        dataframe[name_column].astype(str).str.casefold() == expected_name.casefold()
    ]


def require_one_exact_match(dataframe, name_column, expected_name, item_type):
    matches = exact_name_matches(dataframe, name_column, expected_name)
    if len(matches) == 0:
        raise LookupError(f'{item_type} "{expected_name}" was not found or is not visible.')
    if len(matches) > 1:
        raise LookupError(
            f'{item_type} "{expected_name}" is ambiguous: {len(matches)} exact matches were returned.'
        )
    return matches.iloc[0]


def uuid_or_name(value, fallback_name):
    if value is None or pd.isna(value):
        return fallback_name
    try:
        return UUID(str(value))
    except ValueError:
        return fallback_name


def dax_table_reference(table_name):
    return "'" + str(table_name).replace("'", "''") + "'"


def concise_error(stage, error):
    text = f"{stage}: {type(error).__name__}: {error}"
    return text[:max_error_characters]


def power_bi_request(method, path, json_body=None):
    token = notebookutils.credentials.getToken("pbi")
    if not token:
        raise RuntimeError("Fabric did not return a Power BI access token.")

    response = requests.request(
        method,
        f"https://api.powerbi.com/v1.0/myorg/{path.lstrip('/')}",
        headers={"Authorization": f"Bearer {token}"},
        json=json_body,
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(
            f"Power BI API {method} {path} returned HTTP {response.status_code}: "
            f"{response.text[:max_error_characters]}"
        )
    return response.json()


def execute_power_bi_dax(workspace_id, dataset_id, query):
    payload = power_bi_request(
        "POST",
        f"groups/{workspace_id}/datasets/{dataset_id}/executeQueries",
        {
            "queries": [{"query": query}],
            "serializerSettings": {"includeNulls": True},
        },
    )
    results = payload.get("results") or []
    tables = results[0].get("tables") or [] if results else []
    return tables[0].get("rows") or [] if tables else []


validation_started_utc = datetime.now(timezone.utc)
print(f"Validation started: {validation_started_utc.isoformat()}")


In [ ]:
# Resolve both workspaces from authenticated listings; never guess workspace IDs.
runtime_context = notebookutils.runtime.context
current_workspace_name = str(runtime_context.get("currentWorkspaceName") or "")
current_workspace_id = str(runtime_context.get("currentWorkspaceId") or "")
current_user_name = str(
    runtime_context.get("userName")
    or runtime_context.get("userId")
    or "Unavailable in scheduled job context"
)

if not current_workspace_name or not current_workspace_id:
    raise RuntimeError(
        "Fabric did not provide the required currentWorkspaceName/currentWorkspaceId runtime context. "
        f"Available keys: {sorted(runtime_context.keys())}"
    )

print(f"Execution identity: {current_user_name}")
print(f"Current workspace: {current_workspace_name} ({current_workspace_id})")

workspaces = fabric.list_workspaces()
workspace_name_column = find_column(
    workspaces,
    ["Name", "Workspace Name", "Display Name"],
    "workspace name",
)
workspace_id_column = find_column(
    workspaces,
    ["Id", "ID", "Workspace Id", "Workspace ID"],
    "workspace ID",
)

source_workspace = require_one_exact_match(
    workspaces,
    workspace_name_column,
    source_workspace_name,
    "Source workspace",
)
target_workspace = require_one_exact_match(
    workspaces,
    workspace_name_column,
    target_workspace_name,
    "Target workspace",
)

source_workspace_id = str(source_workspace[workspace_id_column])
target_workspace_id = str(target_workspace[workspace_id_column])
source_workspace_reference = uuid_or_name(source_workspace_id, source_workspace_name)

workspace_results = pd.DataFrame(
    [
        {
            "Role": "Source",
            "Workspace": source_workspace_name,
            "Workspace ID": source_workspace_id,
            "Visible": True,
        },
        {
            "Role": "Target",
            "Workspace": target_workspace_name,
            "Workspace ID": target_workspace_id,
            "Visible": True,
        },
    ]
)
display(workspace_results)

if require_target_workspace_context and current_workspace_name.casefold() != target_workspace_name.casefold():
    raise RuntimeError(
        f'This notebook must run in target workspace "{target_workspace_name}", '
        f'but the current workspace is "{current_workspace_name}". Import or move the notebook '
        "to the target workspace before continuing."
    )


In [ ]:
# Resolve all expected semantic models before attempting metadata or data reads.
datasets = fabric.list_datasets(workspace=source_workspace_reference)
resolved_models = {}
discovery_errors = {}

if datasets.empty or len(datasets.columns) == 0:
    try:
        groups = power_bi_request("GET", "groups?$top=5000").get("value") or []
        group_matches = [
            group
            for group in groups
            if str(group.get("name", "")).casefold() == source_workspace_name.casefold()
        ]
        if len(group_matches) != 1:
            raise LookupError(
                f'Expected one Power BI workspace named "{source_workspace_name}"; '
                f"found {len(group_matches)}."
            )

        power_bi_workspace_id = str(group_matches[0]["id"])
        power_bi_models = power_bi_request(
            "GET", f"groups/{power_bi_workspace_id}/datasets"
        ).get("value") or []

        for semantic_model_name in semantic_model_names:
            matches = [
                model
                for model in power_bi_models
                if str(model.get("name", "")).casefold() == semantic_model_name.casefold()
            ]
            if len(matches) != 1:
                raise LookupError(
                    f'Expected one semantic model named "{semantic_model_name}"; '
                    f"found {len(matches)}."
                )
            dataset_id = str(matches[0]["id"])
            resolved_models[semantic_model_name] = {
                "id": dataset_id,
                "reference": uuid_or_name(dataset_id, semantic_model_name),
                "workspace_id": power_bi_workspace_id,
                "discovery_method": "Power BI REST",
            }
    except Exception as error:
        message = concise_error("Power BI discovery", error)
        discovery_errors.update({name: message for name in semantic_model_names})
else:
    try:
        dataset_name_column = find_column(
            datasets,
            ["Dataset Name", "Name", "Display Name", "Semantic Model Name"],
            "semantic model name",
        )
        dataset_id_column = find_column(
            datasets,
            ["Dataset ID", "Dataset Id", "Id", "ID", "Semantic Model ID"],
            "semantic model ID",
        )
    except Exception as error:
        message = concise_error("Discovery schema", error)
        discovery_errors.update({name: message for name in semantic_model_names})
    else:
        for semantic_model_name in semantic_model_names:
            try:
                dataset_row = require_one_exact_match(
                    datasets,
                    dataset_name_column,
                    semantic_model_name,
                    "Semantic model",
                )
                dataset_id = str(dataset_row[dataset_id_column])
                resolved_models[semantic_model_name] = {
                    "id": dataset_id,
                    "reference": uuid_or_name(dataset_id, semantic_model_name),
                    "workspace_id": source_workspace_id,
                    "discovery_method": "Semantic Link",
                }
            except Exception as error:
                discovery_errors[semantic_model_name] = concise_error("Discovery", error)

discovery_results = pd.DataFrame(
    [
        {
            "Semantic Model": semantic_model_name,
            "Semantic Model ID": resolved_models.get(semantic_model_name, {}).get("id"),
            "Discovery Method": resolved_models.get(semantic_model_name, {}).get("discovery_method"),
            "Discovered": semantic_model_name in resolved_models,
            "Error": discovery_errors.get(semantic_model_name),
        }
        for semantic_model_name in semantic_model_names
    ]
)
display(discovery_results)


In [ ]:
# Run every check for every model and preserve all errors for one complete readiness report.
access_results = []
table_inventory_frames = []

for semantic_model_name in semantic_model_names:
    result = {
        "Semantic Model": semantic_model_name,
        "Semantic Model ID": None,
        "Discovery Method": None,
        "Discovered": False,
        "DAX Connection": False,
        "Metadata Access": False,
        "Data Sample": False,
        "Table Count": None,
        "Column Count": None,
        "Sample Table": None,
        "Status": "NOT READY",
        "Errors": [],
    }

    model = resolved_models.get(semantic_model_name)
    if model is None:
        result["Errors"].append(
            discovery_errors.get(semantic_model_name, "Discovery failed without an error detail.")
        )
        access_results.append(result)
        continue

    dataset_reference = model["reference"]
    result["Semantic Model ID"] = model["id"]
    result["Discovery Method"] = model["discovery_method"]
    result["Discovered"] = True

    try:
        if model["discovery_method"] == "Power BI REST":
            dax_check = execute_power_bi_dax(
                model["workspace_id"],
                model["id"],
                'EVALUATE ROW("__AccessCheck", 1)',
            )
        else:
            dax_check = fabric.evaluate_dax(
                dataset_reference,
                'EVALUATE ROW("__AccessCheck", 1)',
                workspace=source_workspace_reference,
            )
        if len(dax_check) != 1:
            raise RuntimeError(f"Expected one DAX access-check row, received {len(dax_check)}.")
        result["DAX Connection"] = True
    except Exception as error:
        result["Errors"].append(concise_error("DAX connection", error))

    tables = None
    try:
        if model["discovery_method"] == "Power BI REST":
            tables = pd.DataFrame(
                execute_power_bi_dax(
                    model["workspace_id"], model["id"], "EVALUATE INFO.VIEW.TABLES()"
                )
            )
            columns = pd.DataFrame(
                execute_power_bi_dax(
                    model["workspace_id"], model["id"], "EVALUATE INFO.VIEW.COLUMNS()"
                )
            )
        else:
            tables = fabric.list_tables(
                dataset_reference,
                workspace=source_workspace_reference,
                include_internal=False,
            )
            columns = fabric.list_columns(
                dataset_reference,
                workspace=source_workspace_reference,
            )
        if tables.empty:
            raise RuntimeError("Semantic Link returned no visible tables.")

        table_name_column = find_column(tables, ["Name", "Table Name"], "table name")
        inventory = tables.copy()
        inventory.insert(0, "Semantic Model", semantic_model_name)
        table_inventory_frames.append(inventory)

        result["Table Count"] = int(tables[table_name_column].astype(str).nunique())
        result["Column Count"] = int(len(columns))
        result["Metadata Access"] = True
    except Exception as error:
        result["Errors"].append(concise_error("TOM metadata", error))

    if result["Metadata Access"] and tables is not None:
        try:
            table_name_column = find_column(tables, ["Name", "Table Name"], "table name")
            hidden_column = find_column(
                tables,
                ["Hidden", "Is Hidden", "IsHidden"],
                "table hidden flag",
                required=False,
            )
            private_column = find_column(
                tables,
                ["Private", "Is Private", "IsPrivate"],
                "table private flag",
                required=False,
            )

            sample_candidates = tables.copy()
            if hidden_column is not None:
                visible_mask = ~sample_candidates[hidden_column].fillna(False).astype(bool)
                if visible_mask.any():
                    sample_candidates = sample_candidates[visible_mask]

            if private_column is not None:
                private_mask = ~sample_candidates[private_column].fillna(False).astype(bool)
                if private_mask.any():
                    sample_candidates = sample_candidates[private_mask]

            non_system_mask = ~sample_candidates[table_name_column].astype(str).str.startswith(
                ("DateTableTemplate_", "LocalDateTable_", "_Measures")
            )
            if non_system_mask.any():
                sample_candidates = sample_candidates[non_system_mask]

            sample_table_name = str(sample_candidates.iloc[0][table_name_column])
            sample_query = (
                f"EVALUATE TOPN({int(sample_rows_per_model)}, "
                f"{dax_table_reference(sample_table_name)})"
            )
            if model["discovery_method"] == "Power BI REST":
                execute_power_bi_dax(
                    model["workspace_id"], model["id"], sample_query
                )
            else:
                fabric.evaluate_dax(
                    dataset_reference,
                    sample_query,
                    workspace=source_workspace_reference,
                )
            result["Sample Table"] = sample_table_name
            result["Data Sample"] = True
        except Exception as error:
            result["Errors"].append(concise_error("Table data sample", error))

    required_checks = [
        result["Discovered"],
        result["DAX Connection"],
        result["Metadata Access"],
        result["Data Sample"],
    ]
    if all(required_checks):
        result["Status"] = "READY"

    access_results.append(result)

for result in access_results:
    result["Errors"] = " | ".join(result["Errors"])

access_results_df = pd.DataFrame(access_results)
display(access_results_df)

if table_inventory_frames:
    table_inventory = pd.concat(table_inventory_frames, ignore_index=True)
    display(table_inventory)


In [ ]:
# Final gate: do not return a success-shaped result when any prerequisite failed.
not_ready = access_results_df[access_results_df["Status"] != "READY"]
validation_finished_utc = datetime.now(timezone.utc)

if not not_ready.empty:
    failed_models = ", ".join(not_ready["Semantic Model"].astype(str).tolist())
    raise RuntimeError(
        "STEP 0 FAILED. The archive solution is not ready for implementation. "
        f"Review the Errors column for: {failed_models}. "
        "Common blockers are missing Build permission, XMLA being disabled, or insufficient "
        "ReadWrite metadata permission for Semantic Link TOM operations."
    )

print("STEP 0 PASSED")
print(f"Validated all {len(semantic_model_names)} semantic models.")
print(f"Source workspace ID: {source_workspace_id}")
print(f"Target workspace ID: {target_workspace_id}")
print(f"Validation finished: {validation_finished_utc.isoformat()}")
print("It is safe to proceed to notebook 01 for Lakehouse provisioning and audit setup.")
